In [0]:
df = spark.read.table("samples.accuweather.forecast_daily_calendar_metric")
df.show(50)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import SparkSession, DataFrame

SOURCE_TABLE = 'samples.accuweather.forecast_daily_calendar_metric'
TARGET_CATALOG = 'hive_metastore'
TARGET_SCHEMA = 'silver'
TARGET_TABLE = 'weather_forecast'

def create_catalog_and_schema(
    spark: SparkSession,
    catalog: str,
    schema: str
) -> None:
    """
    Create Unity Catalog Data Objects if not present
    args:
        spark: SparkSession
        catalog: str
        schema: str
    """
    spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")


def extract_weather_data(
    spark: SparkSession,
    source_table: str
) -> DataFrame:
    """
    Extract data from source table
    args:
        spark: SparkSession
        source_table: str
    returns:
        DataFrame
    """
    return spark.table(source_table)


def select_columns(
    df: DataFrame
) -> DataFrame:
    """
    Select columns from DataFrame
    args:
        df: DataFrame
    returns:
        DataFrame
    """
    return df.select("*")

def run_pipeline(
    spark: SparkSession
):
    create_catalog_and_schema(spark, TARGET_CATALOG, TARGET_SCHEMA)
    df = extract_weather_data(spark, SOURCE_TABLE)
    df = select_columns(df)
    df.write.mode("overwrite").saveAsTable(f"{TARGET_CATALOG}.{TARGET_SCHEMA}.{TARGET_TABLE}")

run_pipeline(spark)


In [0]:
%sql
SELECT * FROM samples.accuweather.forecast_daily_calendar_imperial
WHERE 
--year(date)=2024 and 
country_code in ('us', 'de', 'jp', 'in', 'ca', 'au', 'nz', 'mx', 'br', 'ar', 'co', 've', 'cl', 'pe', 'uy', 'ec', 'py', 'do', 'gt', 'sv')
LIMIT 1000